## Verify HarmBench compliant cases (olmo2-7b-instruct, contextual)\n
\n
This notebook reproduces the compliant (hb_label=1) count from the labeled JSON output and lists the corresponding record ids / BehaviorIDs.\n
\n
Inputs:\n
- `../data/olmo2_7b_instruct/harmbench_contextual_labeled.json`\n

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("..").resolve()
LABELED_PATH = REPO_ROOT / "data" / "olmo2_7b_instruct" / "harmbench_contextual_labeled.json"

assert LABELED_PATH.is_file(), f"Missing labeled file: {LABELED_PATH}"

with LABELED_PATH.open("r", encoding="utf-8") as f:
    records: list[dict] = json.load(f)

print(f"Loaded records: {len(records)}")
print(f"Records with hb_label: {sum(1 for r in records if 'hb_label' in r)}")


In [ ]:
def summarize(records: list[dict]) -> dict:
    total = len(records)
    with_label = sum(1 for r in records if "hb_label" in r)
    compliant = sum(1 for r in records if r.get("hb_label") == 1)
    non_compliant = sum(1 for r in records if r.get("hb_label") == 0)
    with_errors = sum(1 for r in records if "error" in r)
    no_label = sum(1 for r in records if "hb_label" not in r and "error" not in r)

    asr_pct = (compliant / with_label * 100.0) if with_label else float("nan")

    return {
        "total": total,
        "with_label": with_label,
        "compliant": compliant,
        "non_compliant": non_compliant,
        "errors": with_errors,
        "no_label": no_label,
        "asr_pct": asr_pct,
    }


def compliant_subset(records: list[dict]) -> list[dict]:
    return [r for r in records if r.get("hb_label") == 1]


stats = summarize(records)
comp = compliant_subset(records)

print("Summary")
print("-" * 40)
for k in ["total", "with_label", "compliant", "non_compliant", "errors", "no_label"]:
    print(f"{k:>12}: {stats[k]}")
print(f"{'asr_pct':>12}: {stats['asr_pct']:.1f}%")

EXPECTED_FROM_LOG = 45
print()
print(f"Expected compliant from log: {EXPECTED_FROM_LOG}")
print(f"Actual compliant from JSON:  {len(comp)}")
assert len(comp) == EXPECTED_FROM_LOG, "Compliant count does not match the log output"


In [ ]:
df = pd.DataFrame(comp)

cols = []
for c in ["id", "config", "hb_label", "model_key", "model", "finish_reason"]:
    if c in df.columns:
        cols.append(c)

# Metadata fields (BehaviorID, categories) are nested under `metadata`
if "metadata" in df.columns:
    df["BehaviorID"] = df["metadata"].apply(lambda m: (m or {}).get("BehaviorID"))
    df["SemanticCategory"] = df["metadata"].apply(lambda m: (m or {}).get("SemanticCategory"))
    df["FunctionalCategory"] = df["metadata"].apply(lambda m: (m or {}).get("FunctionalCategory"))
    cols.extend([c for c in ["BehaviorID", "SemanticCategory", "FunctionalCategory"] if c in df.columns])

cols = list(dict.fromkeys(cols))

print(f"Compliant rows: {len(df)}")
print("Unique BehaviorIDs:", df["BehaviorID"].nunique(dropna=True) if "BehaviorID" in df.columns else "n/a")

# Display a compact table for verification (no full prompt/response printed)
df_view = df[cols].sort_values("id") if cols else df.sort_values("id")
df_view
